**Geochemistry Biplot App for Bruker Results.csv Files**

Voila App Version
N. Tripcevich 2026, CC BY-SA 4.0  
[More Information on our Github Repository](https://github.com/arf-berkeley/bruker-xrf-ppm-plot)


In [ ]:
%%capture
%pip install plotly ipywidgets voila

import re
import os
import sys
import functools
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML
import ipywidgets as widgets
from io import StringIO
import base64
from types import SimpleNamespace

# ── Shared state object ────────────────────────────────────────────────────
# SimpleNamespace allows attribute access (state.raw)
# All cells read and write to this object so data flows between tabs
state = SimpleNamespace(
    raw=None,           # raw parsed dataframe direct from CSV
    filtered=None,      # after application/batch filter applied
    study=None,         # fully cleaned PPM dataframe for plotting
    element_cols=None,  # list of element column names after cleaning
    plot_df=None,       # plotly-safe copy of study dataframe
    hover_cols=None,    # columns to show in plot hover tooltip
    name_order=None,    # sorted list of unique sample names
)

#Constants
ALL_METHODS = 'All methods'

# ── Detection limit strings ────────────────────────────────────────────────
# Any value matching these strings is treated as below detection limit
# and will be replaced with 0 during cleaning
LOD_STRINGS = {'< LOD', '<LOD', 'LOD', 'below LOD', 'BDL'}

# ── Filter dropdown sentinel labels ───────────────────────────────────────
ALL_APPS    = 'All applications'
ALL_BATCHES = 'All'

# ── Metadata column list ───────────────────────────────────────────────────
# These columns are dropped during cleaning in Cell 3
# Add any additional Bruker metadata columns here if needed
META_COLS = [
    'Alloy 1', 'Match Qual 1', 'Alloy 2', 'Match Qual 2',
    'Alloy 3', 'Match Qual 3', 'Multiplier', 'Cal Check',
    'Operator', 'Field1', 'Field2', 'ID', '_batch', '_method'
]

# ── Element column pattern ─────────────────────────────────────────────────
# Matches standard element symbol column names e.g. Ti, Cr, Fe, Zr, Au, Pd
# One uppercase letter followed by up to two lowercase letters
# This excludes metadata columns like File # and ElapsedTime
ELEMENT_PATTERN = re.compile(r'^[A-Z][a-z]{0,2}$')

# ── PPM conversion scale ───────────────────────────────────────────────────
# Bruker reports values in weight % so multiply by 10000 to get PPM
PPM_SCALE = 10000

# ── Non-element column list ────────────────────────────────────────────────
# Used by get_elements() in plot cells to exclude metadata from dropdowns
NON_ELEMENT = [
    'File #', 'DateTime', 'Name', 'Application',
    'Method', 'ElapsedTime', 'Elapsed', '_batch', '_method'
]
# ── Default element selections ─────────────────────────────────────────────
# Default X and Y elements for the biplot dropdowns
# Change these to any element symbol you prefer
DEFAULT_BIPLOT_X = 'Sr'
DEFAULT_BIPLOT_Y = 'Rb'

# Default A, B, C elements for the ternary plot dropdowns
DEFAULT_TERNARY_A = 'Rb'
DEFAULT_TERNARY_B = 'Sr'
DEFAULT_TERNARY_C = 'Zr'



**Select the Results.csv table from your Bruker analysis**

Browse to a copy of the __Results.csv__ file typically found in Bruker/Data/Results.csv

Importing all of the Weight Percent data from selected Methods. Batches correspond Application changes while using the XRF.

In [ ]:
# ============================================================
# Cell 1 - Upload and Filter
# Supports: VS Code (local) | Binder | Voila
# ============================================================

# ── Pre-display output container at cell level for Voila ──────────────────
main_out  = widgets.Output()
parse_out = widgets.Output()
display(widgets.HTML('<h3>Step 1 - Upload and Filter</h3>'))
display(main_out)

# ── Environment detection ─────────────────────────────────────────────────
@functools.lru_cache(maxsize=None)
def is_local():
    hosted = [
        'BINDER_LAUNCH_HOST', 'JUPYTERHUB_USER',
        'JUPYTERHUB_SERVICE_PREFIX', 'COLAB_BACKEND_VERSION',
        'VOILA_APP_PORT', 'SERVER_SOFTWARE',
    ]
    if any(os.environ.get(v) for v in hosted):
        return False
    if 'voila' in sys.modules:
        return False
    try:
        import tkinter as tk
        root = tk.Tk()
        root.withdraw()
        root.destroy()
        return True
    except Exception:
        return False

# ── Parser ────────────────────────────────────────────────────────────────
# Uses csv.reader throughout to correctly handle quoted fields
# e.g. "Karaña, site 2" will not be split on the comma
def parse_bruker_results(content_str, verbose=True):
    import csv as _csv

    def _log(*args):
        if verbose:
            with parse_out:
                print(*args)

    # Normalize line endings across Windows/Mac/Linux
    content_str = content_str.replace('\r\n', '\n').replace('\r', '\n')

    # Parse all lines at once using csv.reader so quoted commas are handled
    # StringIO lets csv.reader treat the string as a file-like object
    reader = _csv.reader(
        StringIO(content_str),
        skipinitialspace=True
    )
    all_rows = [
        [c.strip() for c in row]
        for row in reader
        if any(c.strip() for c in row)  # skip completely empty rows
    ]

    # Split into segments — each segment starts with a "File #" header row
    segments        = []
    current_headers = None
    current_rows    = []

    for cols in all_rows:
        if not cols:
            continue

        if cols[0] == 'File #':
            # Save previous segment if it has data rows
            if current_headers and current_rows:
                segments.append((current_headers, current_rows))
                current_rows = []
            current_headers = cols
            continue

        if current_headers:
            current_rows.append(cols)

    # Don't forget the last segment
    if current_headers and current_rows:
        segments.append((current_headers, current_rows))

    if not segments:
        raise ValueError(
            'No segments found. '
            'Expected CSV with "File #" header rows.'
        )

    frames = []
    for seg_idx, (headers, rows) in enumerate(segments, start=1):
        records = []
        for row in rows:
            # Pad short rows or trim long rows to match header length
            if len(row) < len(headers):
                row = row + [''] * (len(headers) - len(row))
            elif len(row) > len(headers):
                row = row[:len(headers)]

            record = {}
            for col, val in zip(headers, row):
                # Normalize below-detection and empty values to NaN
                if val in LOD_STRINGS or val == '':
                    record[col] = np.nan
                else:
                    # Try to parse as float, otherwise keep as string
                    # This preserves diacritics in names like Karaña
                    try:
                        record[col] = float(val)
                    except ValueError:
                        record[col] = val
            records.append(record)

        if records:
            df_seg           = pd.DataFrame(records)
            df_seg['_batch'] = seg_idx
            frames.append(df_seg)

    if not frames:
        raise ValueError('No data rows found after parsing.')

    df = pd.concat(frames, ignore_index=True, join='outer')

    # Re-number batches by Application change so each application
    # transition starts a new batch regardless of segment boundaries
    if 'Application' in df.columns:
        app_series   = df['Application'].fillna('').astype(str)
        df['_batch'] = (app_series != app_series.shift()).cumsum()

    apps = (
        df['Application'].dropna().unique().tolist()
        if 'Application' in df.columns else []
    )
    _log(
        f'Loaded {len(df)} rows | '
        f'{df["_batch"].nunique()} batches | '
        f'Applications: {apps}'
    )
    return df

# ── Upload decoder ────────────────────────────────────────────────────────
# Tries encodings in order most likely for Bruker instrument output.
# Windows-1252 preserves diacritics like ñ (e.g. Karaña) from older
# Windows-based Bruker instruments that do not save as UTF-8.
def _decode_upload(upload_widget):
    val = upload_widget.value
    if isinstance(val, dict):
        if not val:
            raise ValueError('No file uploaded.')
        content_data = next(iter(val.values()))['content']
    elif isinstance(val, (list, tuple)):
        if not val:
            raise ValueError('No file uploaded.')
        content_data = val[0]['content']
    else:
        raise ValueError(f'Unrecognised upload type: {type(val)}')

    if isinstance(content_data, memoryview):
        raw = bytes(content_data)
    else:
        raw = bytes(content_data)

    # Try encodings in order — stop as soon as File # is found in the result
    for encoding in ('utf-8-sig', 'utf-8', 'windows-1252', 'latin-1'):
        try:
            text = raw.decode(encoding)
            if 'File #' in text:
                return text
        except (UnicodeDecodeError, ValueError):
            continue

    # Absolute fallback — latin-1 never raises on any byte sequence
    return raw.decode('latin-1', errors='replace')

# ── Filter UI ─────────────────────────────────────────────────────────────
def build_filter_ui(host_out):
    if state.raw is None:
        with host_out:
            print('No data loaded.')
        return

    state.filtered = state.raw.copy()

    def get_unique(col):
        # Return sorted unique non-empty values for a given column
        if col not in state.raw.columns:
            return []
        return sorted(
            state.raw[col].dropna().astype(str)
                 .str.strip().replace('', np.nan)
                 .dropna().unique().tolist()
        )

    def batches_for(application, method):
        # Return batch numbers that match both the selected application
        # and the selected method
        df = state.raw.copy()
        if application != ALL_APPS:
            df = df[
                df['Application'].astype(str).str.strip()
                == application.strip()
            ]
        if method != ALL_METHODS:
            df = df[
                df['Method'].astype(str).str.strip()
                == method.strip()
            ]
        return [ALL_BATCHES] + [
            str(b) for b in sorted(df['_batch'].dropna().unique())
        ]

    def methods_for(application):
        # Return method names relevant to the selected application only
        df = state.raw.copy()
        if application != ALL_APPS:
            df = df[
                df['Application'].astype(str).str.strip()
                == application.strip()
            ]
        if 'Method' not in df.columns:
            return [ALL_METHODS]
        return [ALL_METHODS] + sorted(
            df['Method'].dropna().astype(str)
              .str.strip().replace('', np.nan)
              .dropna().unique().tolist()
        )

    all_apps    = [ALL_APPS] + get_unique('Application')
    default_app = all_apps[1] if len(all_apps) > 1 else ALL_APPS

    app_dd = widgets.Dropdown(
        options=all_apps,
        value=default_app,
        description='Application:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='360px')
    )
    method_dd = widgets.Dropdown(
        options=methods_for(default_app),
        value=ALL_METHODS,
        description='Method:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='280px')
    )
    batch_dd = widgets.Dropdown(
        options=batches_for(default_app, ALL_METHODS),
        value=ALL_BATCHES,
        description='Batch:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='180px')
    )
    apply_btn = widgets.Button(
        description='Apply Filter',
        button_style='primary',
        icon='filter',
        layout=widgets.Layout(width='150px')
    )
    summary_out = widgets.Output()
    filter_out  = widgets.Output()

    def refresh_summary(application, method):
        # Show row count, batch list and date range for current selection
        with summary_out:
            summary_out.clear_output(wait=True)
            df = state.raw.copy()
            if application != ALL_APPS:
                df = df[
                    df['Application'].astype(str).str.strip()
                    == application.strip()
                ]
            if method != ALL_METHODS:
                df = df[
                    df['Method'].astype(str).str.strip()
                    == method.strip()
                ]
            batches = sorted(df['_batch'].unique().tolist())
            dates   = pd.to_datetime(
                df['DateTime'], errors='coerce'
            ).dropna()
            d_min = dates.min().strftime('%m-%d-%Y') if not dates.empty else '?'
            d_max = dates.max().strftime('%m-%d-%Y') if not dates.empty else '?'
            print(f'  Application : {application}')
            print(f'  Method      : {method}')
            print(f'  Rows        : {len(df)}')
            print(f'  Batches     : {batches}')
            print(f'  Dates       : {d_min} - {d_max}')

    def on_app_change(change):
        # When application changes update method and batch dropdowns
        new_methods      = methods_for(change['new'])
        method_dd.options = new_methods
        method_dd.value   = ALL_METHODS
        batch_dd.options  = batches_for(change['new'], ALL_METHODS)
        batch_dd.value    = ALL_BATCHES
        refresh_summary(change['new'], ALL_METHODS)

    def on_method_change(change):
        # When method changes update batch dropdown
        batch_dd.options = batches_for(app_dd.value, change['new'])
        batch_dd.value   = ALL_BATCHES
        refresh_summary(app_dd.value, change['new'])

    def on_apply(b):
        filter_out.clear_output(wait=True)
        with filter_out:
            df        = state.raw.copy()
            sel_app    = app_dd.value
            sel_method = method_dd.value
            sel_batch  = batch_dd.value

            # Apply application filter
            if sel_app != ALL_APPS:
                df = df[
                    df['Application'].astype(str).str.strip()
                    == sel_app.strip()
                ]

            # Apply method filter
            if sel_method != ALL_METHODS:
                df = df[
                    df['Method'].astype(str).str.strip()
                    == sel_method.strip()
                ]

            # Apply batch filter
            if sel_batch != ALL_BATCHES:
                try:
                    df = df[df['_batch'] == int(sel_batch)]
                except ValueError:
                    print(f'Invalid batch: {sel_batch}')
                    return

            df             = df.reset_index(drop=True)
            state.filtered = df

            if df.empty:
                print(
                    f'No rows for app="{sel_app}" '
                    f'method="{sel_method}" '
                    f'batch="{sel_batch}"'
                )
                return

            dates = pd.to_datetime(
                df['DateTime'], errors='coerce'
            ).dropna()
            d_min = dates.min().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'
            d_max = dates.max().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'

            try:
                file_nums    = df['File #'].dropna().astype(int)
                f_min, f_max = int(file_nums.min()), int(file_nums.max())
            except Exception:
                f_min = f_max = '?'

            print(f'✓ {len(df)} rows kept')
            print(f'  Application : {sel_app}')
            print(f'  Method      : {sel_method}')
            print(f'  Batch       : {sel_batch}')
            print(f'  File #      : {f_min} - {f_max}')
            print(f'  Dates       : {d_min} - {d_max}')
            print('\nProceed to Step 2 Clean Data.')

    app_dd.observe(on_app_change, names='value')
    method_dd.observe(on_method_change, names='value')
    apply_btn.on_click(on_apply)

    with host_out:
        host_out.clear_output(wait=True)
        display(parse_out)
        display(widgets.HTML(
            '<b>Filter by Application, Method and Batch</b><br>'
            '<span style="color:grey;font-size:12px">'
            'Select an Application — Method and Batch update automatically. '
            'Then click Apply Filter.</span>'
        ))
        display(widgets.VBox([
            widgets.HBox([app_dd, method_dd, batch_dd, apply_btn]),
            summary_out,
            filter_out
        ]))
    refresh_summary(app_dd.value, ALL_METHODS)

# ── Load local ────────────────────────────────────────────────────────────
def load_local():
    import tkinter as tk
    from tkinter import filedialog
    if state.raw is not None:
        with main_out:
            print(f'Already loaded: {state.raw.shape[0]} rows')
        build_filter_ui(main_out)
        return
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    path = filedialog.askopenfilename(
        title='Select Results.csv',
        filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
    )
    root.destroy()
    with main_out:
        if path:
            try:
                # Try encodings in order to preserve diacritics like ñ
                content = None
                for encoding in ('utf-8-sig', 'utf-8', 'windows-1252', 'latin-1'):
                    try:
                        with open(path, 'r', encoding=encoding) as f:
                            text = f.read()
                        if 'File #' in text:
                            content = text
                            break
                    except (UnicodeDecodeError, ValueError):
                        continue
                if content is None:
                    with open(path, 'r', encoding='latin-1', errors='replace') as f:
                        content = f.read()
                state.raw = parse_bruker_results(content)
                print(f'Loaded: {path}')
                build_filter_ui(main_out)
            except Exception as e:
                print(f'Failed to load: {e}')
        else:
            print('No file selected. Re-run cell to try again.')

# ── Load hosted ───────────────────────────────────────────────────────────
def load_hosted():
    if state.raw is not None:
        with main_out:
            print(f'Already loaded: {state.raw.shape[0]} rows')
        build_filter_ui(main_out)
        return

    upload_widget = widgets.FileUpload(accept='.csv', multiple=False)
    load_btn      = widgets.Button(
        description='Load Data',
        button_style='success',
        icon='check',
        disabled=True
    )
    status_lbl  = widgets.Label('Upload a Bruker XRF Results.csv file')
    session_lbl = widgets.HTML(
        '<span style="color:grey;font-size:12px">'
        'Sessions are temporary - re-upload each session.</span>'
    )
    filter_area = widgets.Output()

    def _on_upload_change(change):
        if upload_widget.value:
            load_btn.disabled = False
            try:
                val  = upload_widget.value
                size = (
                    next(iter(val.values())).get('size', 0)
                    if isinstance(val, dict)
                    else len(val[0].get('content', b''))
                )
                size_mb          = size / (1024 * 1024)
                status_lbl.value = (
                    f'Large file ({size_mb:.1f} MB) - may be slow. Click Load Data.'
                    if size_mb > 50
                    else f'File ready ({size_mb:.1f} MB) - click Load Data'
                )
            except Exception:
                status_lbl.value = 'File ready - click Load Data'
        else:
            load_btn.disabled = True

    def _on_load(b):
        try:
            content              = _decode_upload(upload_widget)
            state.raw            = parse_bruker_results(content)
            status_lbl.value     = f'Loaded {state.raw.shape[0]} rows'
            load_btn.disabled    = True
            load_btn.description = 'Loaded'
            with filter_area:
                filter_area.clear_output(wait=True)
                build_filter_ui(filter_area)
        except Exception as e:
            status_lbl.value = f'Error: {e}'

    upload_widget.observe(_on_upload_change, names='value')
    load_btn.on_click(_on_load)

    with main_out:
        display(widgets.VBox([
            widgets.Label('Upload Results.csv:'),
            upload_widget,
            load_btn,
            status_lbl,
            session_lbl,
            filter_area
        ]))

# ── Entry point ───────────────────────────────────────────────────────────
if is_local():
    load_local()
else:
    load_hosted()

Cleaning data includes removing the following: elemental error columns, Alloy, Match Qual columns, Multiplier, Cal Check, Operator, Field 1&2. This script also replaces Below Detection Limits LOD with 0.

You may now select rows of data from recent analyses by filtering with either File # or Date.

In [ ]:
# ============================================================
# Cell 2 - Clean Data
# Supports: VS Code (local) | Binder | Voila
# ============================================================

cell3_out = widgets.Output()
clean_btn = widgets.Button(
    description='Clean Data',
    button_style='primary',
    icon='cog',
    layout=widgets.Layout(width='160px')
)

display(widgets.HTML('<h3>Step 2 - Clean Data</h3>'))
display(clean_btn)
display(cell3_out)

def on_clean(btn):
    cell3_out.clear_output(wait=True)
    with cell3_out:
        if state.filtered is None or len(state.filtered) == 0:
            print('Please upload and filter data in Step 1 first.')
            return

        _source = state.filtered.copy()
        print(f'Using filtered data : {len(_source)} rows')
        print(f'Application(s)      : '
              f'{_source["Application"].dropna().unique().tolist()}')
        print(f'Batch(es)           : '
              f'{sorted(_source["_batch"].unique().tolist())}')

        # ── Drop non-element columns ───────────────────────────────────────
        drop_cols = [
            c for c in _source.columns
            if c in META_COLS or 'Err' in c
        ]
        keep_cols = [c for c in _source.columns if c not in drop_cols]
        study     = _source[keep_cols].copy()

        # ── Identify column types ──────────────────────────────────────────
        string_cols = [
            c for c in ['Name', 'Application', 'Method']
            if c in study.columns
        ]
        meta_numeric = ['File #', 'ElapsedTime']

        element_cols = [
            c for c in study.columns
            if ELEMENT_PATTERN.match(c)
            and c not in string_cols
            and c not in meta_numeric
            and c != 'DateTime'
        ]
        numeric_cols = [
            c for c in study.columns
            if c not in string_cols
            and c != 'DateTime'
        ]

        # ── Type casting ───────────────────────────────────────────────────
        study[string_cols]  = study[string_cols].astype('string')
        study[numeric_cols] = study[numeric_cols].apply(
            pd.to_numeric, errors='coerce'
        )
        if 'DateTime' in study.columns:
            study['DateTime'] = pd.to_datetime(
                study['DateTime'], errors='coerce'
            )

        # ── Scale to ppm and fill LOD with 0 ──────────────────────────────
        study[element_cols] = (study[element_cols] * PPM_SCALE).round(1)
        study[element_cols] = study[element_cols].fillna(0)

        # ── Drop empty rows and columns ────────────────────────────────────
        study.dropna(axis=1, how='all', inplace=True)
        study.dropna(how='all', inplace=True)
        study = study.reset_index(drop=True)

        # ── Re-identify element cols after dropna ──────────────────────────
        element_cols = [c for c in element_cols if c in study.columns]

        # ── Build plot-ready dataframe ─────────────────────────────────────
        plot_df = study.copy()
        for col in plot_df.columns:
            if pd.api.types.is_string_dtype(plot_df[col]) or \
               pd.api.types.is_object_dtype(plot_df[col]):
                plot_df[col] = (
                    plot_df[col]
                    .astype(object)
                    .fillna('(no name)')
                    .astype(str)
                    .replace({
                        'nan'  : '(no name)',
                        'None' : '(no name)',
                        '<NA>' : '(no name)',
                        ''     : '(no name)'
                    })
                )

        if 'Name' not in plot_df.columns:
            plot_df['Name'] = '(no name)'
        else:
            plot_df['Name'] = (
                plot_df['Name']
                .astype(str)
                .replace({
                    'nan'  : '(no name)',
                    'None' : '(no name)',
                    '<NA>' : '(no name)',
                    ''     : '(no name)'
                })
            )

        for col in element_cols:
            plot_df[col] = pd.to_numeric(
                plot_df[col], errors='coerce'
            ).fillna(0)

        if 'DateTime' in plot_df.columns:
            plot_df['DateTime'] = (
                pd.to_datetime(plot_df['DateTime'], errors='coerce')
                .dt.strftime('%m-%d-%Y %H:%M')
                .fillna('unknown')
            )

        if 'File #' in plot_df.columns:
            plot_df['File #'] = (
                pd.to_numeric(plot_df['File #'], errors='coerce')
                .fillna(0)
                .astype(int)
                .astype(str)
            )

        hover_cols = [
            c for c in ['File #', 'DateTime', 'Application']
            if c in plot_df.columns
        ]
        name_order = sorted(plot_df['Name'].unique().tolist())

        # ── Store on state ─────────────────────────────────────────────────
        state.study        = study
        state.element_cols = element_cols
        state.plot_df      = plot_df
        state.hover_cols   = hover_cols
        state.name_order   = name_order

        # ── Summary ────────────────────────────────────────────────────────
        print(f'\nRows after cleaning : {len(study)}')
        print(f'Element columns     : {element_cols}')
        print(f'Name values         : {name_order}')
        print('\nProceed to Step 3 Biplot or Step 4 Ternary.')

clean_btn.on_click(on_clean)

In [ ]:
# ============================================================
# Cell 3 - Biplot
# Supports: VS Code (local) | Binder | Voila
# ============================================================

cell4_out   = widgets.Output()
biplot_btn  = widgets.Button(
    description='Draw Biplot',
    button_style='success',
    icon='bar-chart',
    layout=widgets.Layout(width='160px')
)
biplot_plot = widgets.Output()

display(widgets.HTML('<h3>Step 3 - Biplot</h3>'))
display(biplot_btn)
display(cell4_out)
display(biplot_plot)

def on_biplot(btn):
    cell4_out.clear_output(wait=True)
    biplot_plot.clear_output(wait=True)
    with cell4_out:
        if state.plot_df is None:
            print('Please run Step 2 Clean Data first.')
            return

        element_cols = state.element_cols
        if not element_cols:
            print('No element columns found. Check Step 2.')
            return

        default_x = (
            DEFAULT_BIPLOT_X if DEFAULT_BIPLOT_X in element_cols
            else element_cols[0]
        )
        default_y = (
            DEFAULT_BIPLOT_Y if DEFAULT_BIPLOT_Y in element_cols
            else element_cols[1] if len(element_cols) > 1
            else element_cols[0]
        )

        x_dd = widgets.Dropdown(
            options=element_cols,
            value=default_x,
            description='X Axis:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px')
        )
        y_dd = widgets.Dropdown(
            options=element_cols,
            value=default_y,
            description='Y Axis:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px')
        )

        def update_biplot(change=None):
            with biplot_plot:
                biplot_plot.clear_output(wait=True)
                x          = x_dd.value
                y          = y_dd.value
                plot_df    = state.plot_df.copy()

                # ── Choose color grouping ──────────────────────────────────
                # Use Application if multiple applications are present
                # so the legend shows meaningful group distinctions.
                # Fall back to Name for single-application datasets
                # where sample name is more informative.
                if (
                    'Application' in plot_df.columns
                    and plot_df['Application'].nunique() > 1
                ):
                    color_col = 'Application'
                elif 'Name' in plot_df.columns:
                    color_col = 'Name'
                else:
                    color_col = None

                # Build sorted category order for the chosen color column
                color_order = (
                    sorted(plot_df[color_col].unique().tolist())
                    if color_col else []
                )

                # Filter to rows with valid positive values for both axes
                plot_df = plot_df[
                    (plot_df[x] > 0) & (plot_df[y] > 0)
                ]
                if plot_df.empty:
                    print(f'No rows with valid data for {x} and {y}.')
                    return

                # ── Build custom hover template ────────────────────────────
                # Name is shown first in bold, DateTime is excluded.
                # File # and Application are shown if present.
                # %{customdata} indices map to the customdata_cols list.
                customdata_cols = []
                if 'Name' in plot_df.columns:
                    customdata_cols.append('Name')
                if 'File #' in plot_df.columns:
                    customdata_cols.append('File #')
                if 'Application' in plot_df.columns:
                    customdata_cols.append('Application')

                # Build the hover template line by line
                hover_lines = []
                for i, col in enumerate(customdata_cols):
                    if col == 'Name':
                        # Name is bold and labelled on first line
                        hover_lines.append(
                            f'<b>Name: %{{customdata[{i}]}}</b>'
                        )
                    else:
                        hover_lines.append(
                            f'{col}: %{{customdata[{i}]}}'
                        )

                # Add x and y axis values
                hover_lines.append(f'{x}: %{{x:.1f}} ppm')
                hover_lines.append(f'{y}: %{{y:.1f}} ppm')

                # Join lines with HTML line break
                hovertemplate = '<br>'.join(hover_lines) + '<extra></extra>'

                try:
                    fig = px.scatter(
                        plot_df, x=x, y=y,
                        color=color_col,
                        category_orders={color_col: color_order},
                        # Pass customdata columns for hover template
                        custom_data=customdata_cols,
                        title=f'{y} vs {x} Biplot  (n={len(plot_df)})',
                        labels={
                            x        : f'{x} (ppm)',
                            y        : f'{y} (ppm)',
                            color_col: color_col
                        }
                    )
                    # Apply custom hover template to all traces
                    fig.update_traces(
                        marker=dict(size=8, opacity=0.85),
                        hovertemplate=hovertemplate
                    )
                    fig.update_layout(
                        height=600,
                        hovermode='closest',
                        legend=dict(
                            title=dict(
                                text=color_col,
                                font=dict(size=13)
                            ),
                            itemsizing='constant',
                            bordercolor='lightgrey',
                            borderwidth=1,
                            bgcolor='rgba(255,255,255,0.85)',
                            x=1.02, xanchor='left',
                            y=1,    yanchor='top'
                        ),
                        margin=dict(r=180)
                    )
                    display(fig)
                except Exception as e:
                    print(f'Plot error: {e}')

        x_dd.observe(update_biplot, names='value')
        y_dd.observe(update_biplot, names='value')

        display(widgets.HBox([x_dd, y_dd]))
        update_biplot()

biplot_btn.on_click(on_biplot)

In [ ]:
# ============================================================
# Cell 4 - Ternary Plot
# Supports: VS Code (local) | Binder | Voila
# ============================================================

cell5_out    = widgets.Output()
ternary_btn  = widgets.Button(
    description='Draw Ternary',
    button_style='success',
    icon='signal',
    layout=widgets.Layout(width='160px')
)
ternary_plot = widgets.Output()

display(widgets.HTML('<h3>Step 4 - Ternary Plot</h3>'))
display(ternary_btn)
display(cell5_out)
display(ternary_plot)

def on_ternary(btn):
    cell5_out.clear_output(wait=True)
    ternary_plot.clear_output(wait=True)
    with cell5_out:
        if state.plot_df is None:
            print('Please run Step 2 Clean Data first.')
            return

        element_cols = state.element_cols
        if len(element_cols) < 3:
            print(f'Need at least 3 elements. Found: {element_cols}')
            return

        default_a = (
            DEFAULT_TERNARY_A if DEFAULT_TERNARY_A in element_cols
            else element_cols[0]
        )
        default_b = (
            DEFAULT_TERNARY_B if DEFAULT_TERNARY_B in element_cols
            else element_cols[1]
        )
        default_c = (
            DEFAULT_TERNARY_C if DEFAULT_TERNARY_C in element_cols
            else element_cols[2]
        )

        a_dd = widgets.Dropdown(
            options=element_cols,
            value=default_a,
            description='A (top):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )
        b_dd = widgets.Dropdown(
            options=element_cols,
            value=default_b,
            description='B (bottom left):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )
        c_dd = widgets.Dropdown(
            options=element_cols,
            value=default_c,
            description='C (bottom right):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )

        def update_ternary(change=None):
            with ternary_plot:
                ternary_plot.clear_output(wait=True)
                a = a_dd.value
                b = b_dd.value
                c = c_dd.value

                if len({a, b, c}) < 3:
                    print('Please select three different elements.')
                    return

                plot_df = state.plot_df.copy()

                # ── Choose color grouping ──────────────────────────────────
                # Use Application if multiple applications are present
                # so the legend shows meaningful group distinctions.
                # Fall back to Name for single-application datasets
                # where sample name is more informative.
                if (
                    'Application' in plot_df.columns
                    and plot_df['Application'].nunique() > 1
                ):
                    color_col = 'Application'
                elif 'Name' in plot_df.columns:
                    color_col = 'Name'
                else:
                    color_col = None

                # Build sorted category order for the chosen color column
                color_order = (
                    sorted(plot_df[color_col].unique().tolist())
                    if color_col else []
                )

                # Filter to rows with valid positive values for all three axes
                plot_df = plot_df[
                    (plot_df[a] > 0) &
                    (plot_df[b] > 0) &
                    (plot_df[c] > 0)
                ].copy()

                if plot_df.empty:
                    print(f'No rows with valid data for {a}, {b}, {c}.')
                    return

                # ── Build custom hover template ────────────────────────────
                # Plotly ternary %{a} %{b} %{c} show normalized proportions
                # not original PPM values. To show PPM we store the original
                # values in customdata and reference them by index instead.
                #
                # customdata column order:
                #   0      : Name       (if present)
                #   1      : File #     (if present)
                #   2      : Application (if present)
                #   last-3 : a PPM value
                #   last-2 : b PPM value
                #   last-1 : c PPM value

                customdata_cols = []
                if 'Name' in plot_df.columns:
                    customdata_cols.append('Name')
                if 'File #' in plot_df.columns:
                    customdata_cols.append('File #')
                if 'Application' in plot_df.columns:
                    customdata_cols.append('Application')

                # Append original PPM columns for a, b, c at the end
                # These are the raw values before ternary normalization
                customdata_cols.append(a)
                customdata_cols.append(b)
                customdata_cols.append(c)

                # Get the indices of a, b, c in customdata
                idx_a = customdata_cols.index(a)
                idx_b = customdata_cols.index(b)
                idx_c = customdata_cols.index(c)

                # Build hover template line by line
                hover_lines = []
                for i, col in enumerate(customdata_cols):
                    if col == 'Name':
                        # Name is bold and on the first line
                        hover_lines.append(
                            f'<b>Name: %{{customdata[{i}]}}</b>'
                        )
                    elif col == a:
                        # Show original PPM value for a axis
                        hover_lines.append(
                            f'{a}: %{{customdata[{idx_a}]:.1f}} ppm'
                        )
                    elif col == b:
                        # Show original PPM value for b axis
                        hover_lines.append(
                            f'{b}: %{{customdata[{idx_b}]:.1f}} ppm'
                        )
                    elif col == c:
                        # Show original PPM value for c axis
                        hover_lines.append(
                            f'{c}: %{{customdata[{idx_c}]:.1f}} ppm'
                        )
                    else:
                        hover_lines.append(
                            f'{col}: %{{customdata[{i}]}}'
                        )

                # Join with HTML line break, suppress default trace box
                hovertemplate = '<br>'.join(hover_lines) + '<extra></extra>'

                try:
                    fig = px.scatter_ternary(
                        plot_df, a=a, b=b, c=c,
                        color=color_col,
                        category_orders={color_col: color_order},
                        # customdata includes Name, File #, Application
                        # plus original PPM values for a, b, c
                        custom_data=customdata_cols,
                        title=f'Ternary: {a} / {b} / {c}  (n={len(plot_df)})',
                        labels={
                            color_col : color_col,
                            a         : f'{a} (ppm)',
                            b         : f'{b} (ppm)',
                            c         : f'{c} (ppm)'
                        }
                    )
                    # Apply custom hover template to all traces
                    fig.update_traces(
                        marker=dict(size=8, opacity=0.85),
                        hovertemplate=hovertemplate
                    )
                    fig.update_layout(
                        height=650,
                        legend=dict(
                            title=dict(
                                text=color_col,
                                font=dict(size=13)
                            ),
                            itemsizing='constant',
                            bordercolor='lightgrey',
                            borderwidth=1,
                            bgcolor='rgba(255,255,255,0.85)',
                            x=1.02, xanchor='left',
                            y=1,    yanchor='top'
                        ),
                        margin=dict(r=180)
                    )
                    display(fig)
                except Exception as e:
                    print(f'Plot error: {e}')

        a_dd.observe(update_ternary, names='value')
        b_dd.observe(update_ternary, names='value')
        c_dd.observe(update_ternary, names='value')

        display(widgets.HBox([a_dd, b_dd, c_dd]))
        update_ternary()

ternary_btn.on_click(on_ternary)

In [ ]:
# ============================================================
# Cell 5 - Export CSV and Restart
# Supports: VS Code (local) | Binder | Voila
# ============================================================
import base64
from IPython.display import HTML as IHTML

cell6_out   = widgets.Output()
export_btn  = widgets.Button(
    description='Export CSV',
    button_style='warning',
    icon='download',
    layout=widgets.Layout(width='160px')
)
restart_btn = widgets.Button(
    description='Restart',
    button_style='danger',
    icon='refresh',
    layout=widgets.Layout(width='160px')
)
restart_out = widgets.Output()
export_link = widgets.Output()

display(widgets.HTML('<h3>Step 5 - Export CSV</h3>'))
display(widgets.HBox([export_btn, restart_btn]))
display(cell6_out)
display(export_link)
display(restart_out)

def on_export(btn):
    cell6_out.clear_output(wait=True)
    export_link.clear_output(wait=True)
    with cell6_out:
        if state.study is None:
            print('Please run Step 2 Clean Data first.')
            return

        df = state.study

        try:
            dates    = pd.to_datetime(
                df['DateTime'], errors='coerce'
            ).dropna()
            date_str = (
                dates.max().strftime('%Y%m%d')
                if not dates.empty else 'export'
            )
        except Exception:
            date_str = 'export'

        try:
            app_str = (
                df['Application']
                .dropna()
                .unique()[0]
                .replace(' ', '_')
                if 'Application' in df.columns
                else 'Bruker'
            )
        except Exception:
            app_str = 'Bruker'

        filename = f'Bruker_{app_str}_{date_str}.csv'
        csv_str  = df.to_csv(index=False)
        b64      = base64.b64encode(csv_str.encode()).decode()

        with export_link:
            display(IHTML(
                f'<a download="{filename}" '
                f'href="data:text/csv;base64,{b64}" '
                f'style="font-size:14px;font-weight:bold;">'
                f'Click here to download {filename}</a>'
            ))

        print(f'Ready     : {filename}')
        print(f'Rows      : {len(df)}')
        print(f'Columns   : {df.columns.tolist()}')

def on_restart(btn):
    with restart_out:
        restart_out.clear_output(wait=True)
        try:
            # ── Reset all state variables ──────────────────────────────────
            # Clears all loaded data, filtered data, cleaned data and
            # plot-ready data so the notebook can be used with a new file
            state.raw          = None
            state.filtered     = None
            state.study        = None
            state.element_cols = None
            state.plot_df      = None
            state.hover_cols   = None
            state.name_order   = None

            # ── Clear all output widgets ───────────────────────────────────
            # This clears the visible output of every cell so the UI
            # returns to its initial state without restarting the kernel
            for out in [
                main_out, parse_out,
                cell3_out,
                cell4_out, biplot_plot,
                cell5_out, ternary_plot,
                cell6_out, export_link,
            ]:
                try:
                    out.clear_output(wait=True)
                except Exception:
                    pass

            # ── Reset environment detection cache ──────────────────────────
            # lru_cache on is_local() must be cleared so it can re-detect
            # the environment after restart
            try:
                is_local.cache_clear()
            except Exception:
                pass

            print('✓ Reset complete — re-upload your Results.csv in Step 1.')

        except Exception as e:
            print(f'Reset error: {e}')

export_btn.on_click(on_export)
restart_btn.on_click(on_restart)